# Hotel booking cancellation — model development

This notebook explains the temporal validation results and the current candidate.
The full experiment log remains in this file's Git history. Training runs through
`src/` and `dvc.yaml`; this notebook reads saved development and CV results.

**Decision status:** RF with the 17-feature F1 set and raw probabilities is the
current candidate for H1 and H2. Final model selection, refit, and out-of-time
test evaluation are separate gates. This notebook does not read test labels.

## 1. Setup and experiment sources

`params.yaml` defines the current pipeline settings. Historical `reports/`
artifacts preserve the comparable model search. `artifacts/` contains the
current DVC split, CV and error-analysis outputs. All reported CV means give
each validation window equal weight.

In [1]:
from pathlib import Path
import json
import pandas as pd
import yaml

def project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'params.yaml').exists() and (candidate / 'Model pipeline').is_dir():
            return candidate
    raise FileNotFoundError('Run from the Project directory or one of its children')

PROJECT = project_root()
MODEL = PROJECT / 'Model pipeline'
REPORTS = MODEL / 'reports'
ARTIFACTS = MODEL / 'artifacts'
params = yaml.safe_load((PROJECT / 'params.yaml').read_text(encoding='utf-8'))
HOTELS = tuple(params['hotels'])
METRICS = ['LogLoss', 'BrierScore', 'ROC_AUC', 'AveragePrecision']
assert (params['model'], params['features'], params['calibration']) == ('rf', 'f1_17', 'raw'), (
    'The candidate configuration changed; revisit this report before using its conclusion')

def report_table(run, filename='fold_metrics.csv'):
    return pd.concat([pd.read_csv(REPORTS / run / hotel / filename,
                                  dtype={'Booking_ID': str}, float_precision='round_trip')
                      for hotel in HOTELS], ignore_index=True)

def show(frame, rows=30):
    print(frame.head(rows).to_string(index=False, float_format=lambda value: f'{value:.4f}'))

print('Hotels:', ', '.join(HOTELS))
print('Current candidate:', params['model'], params['features'], params['calibration'])
print('Validation folds:', params['split']['n_folds'])

Hotels: H1, H2
Current candidate: rf f1_17 raw
Validation folds: 4


The pipeline records source dependencies and parameters in DVC. The historical
search artifacts are retained because they document experiments beyond the
current candidate. `tools/check_legacy_parity.py` passes with documented,
metric-specific replay tolerances. The table below shows the remaining tiny
numeric differences. The model-comparison table uses **one historical result
set** throughout.

In [2]:
parity_rows = []
for hotel in HOTELS:
    current = pd.read_csv(ARTIFACTS / 'tuning' / hotel / 'fold_metrics.csv')
    previous = pd.read_csv(REPORTS / 'tuning_round1_v1' / hotel / 'fold_metrics.csv')
    joined = current.merge(previous, on=['ConfigID', 'Fold'], how='outer',
                           suffixes=('_dvc', '_historical'), indicator=True,
                           validate='one_to_one')
    assert joined._merge.eq('both').all() and len(joined) == len(previous)
    for family in ('lr', 'rf', 'xgb'):
        rows = joined.loc[joined.ConfigID.str.startswith(family)]
        checked_metrics = METRICS + [f'Train{name}' for name in METRICS] + [
            'DummyLogLoss', 'GainVsDummy']
        parity_rows.append({
            'Hotel': hotel, 'Family': family.upper(), 'ConfigFoldRows': len(rows),
            'MaxAbsLogLossDifference': (rows.LogLoss_dvc - rows.LogLoss_historical).abs().max(),
            'MaxAbsMetricDifference': max((rows[f'{name}_dvc'] - rows[f'{name}_historical'])
                                          .abs().max() for name in checked_metrics),
        })
parity = pd.DataFrame(parity_rows)
print(parity.to_string(index=False, formatters={
    'MaxAbsLogLossDifference': '{:.2e}'.format,
    'MaxAbsMetricDifference': '{:.2e}'.format,
}))

Hotel Family  ConfigFoldRows MaxAbsLogLossDifference MaxAbsMetricDifference
   H1     LR              16                4.91e-10               6.16e-07
   H1     RF              36                5.55e-17               1.11e-16
   H1    XGB              48                0.00e+00               0.00e+00
   H2     LR              16                7.37e-10               2.47e-07
   H2     RF              36                1.11e-16               1.11e-16
   H2    XGB              48                0.00e+00               0.00e+00


## 2. Temporal split and label eligibility

Four two-month validation windows precede the held-out test window. A booking
can enter training only if `CutoffDate < FitDate` and its label was available
**strictly before** that fit date. For `Check-Out`, availability is the later of
`ReservationStatusDate` and expected checkout (`ArrivalDate + TotalNights`).
The preprocessor is fitted afresh on each eligible training cohort.

In [3]:
import filecmp
split_files = ('partition_membership.csv', 'cv_membership.csv',
               'fit_eligibility.csv', 'refit_membership.csv',
               'calibration_membership.csv')
split_summaries = []
partition_summaries = []
for hotel in HOTELS:
    for filename in split_files:
        assert filecmp.cmp(ARTIFACTS / 'split' / hotel / filename,
                           MODEL / 'splits' / 'split_v1' / hotel / filename,
                           shallow=False), f'{hotel}: split differs from historical reports'
    audit = json.loads((ARTIFACTS / 'split' / hotel / 'audit.json').read_text(encoding='utf-8'))
    assert all(check['Passed'] for check in audit['checks']), f'{hotel}: split audit failed'
    split_summaries.extend(audit['cv_summary'])
    partition_summaries.extend(audit['partition_summary'])
split_summary = pd.DataFrame(split_summaries)
partition_summary = pd.DataFrame(partition_summaries)
show(split_summary[['Hotel', 'Fold', 'FitDate', 'TrainCandidates', 'TrainRows',
                    'TrainLabelsUnavailable', 'ValidationRows', 'ValidationPositives']])

Hotel  Fold             FitDate  TrainCandidates  TrainRows  TrainLabelsUnavailable  ValidationRows  ValidationPositives
   H1     1 2016-07-01T00:00:00             9077       8618                     459            1779                  115
   H1     2 2016-09-01T00:00:00            10856      10353                     503            1936                  112
   H1     3 2016-11-01T00:00:00            12792      12291                     501            1152                  114
   H1     4 2017-01-01T00:00:00            13944      13631                     313            1252                  119
   H2     1 2016-07-01T00:00:00            15188      14230                     958            3805                  316
   H2     2 2016-09-01T00:00:00            18993      17962                    1031            3958                  534
   H2     3 2016-11-01T00:00:00            22951      22069                     882            2216                  293
   H2     4 2017-01-01T00:00:00 

In [4]:
show(partition_summary[['Hotel', 'Cohort', 'Rows', 'Positives']])

Hotel               Cohort  Rows  Positives
   H1          Development 16564  1269.0000
   H1 Final refit eligible 16019  1250.0000
   H1 Final refit excluded   545    19.0000
   H1          Calibration  6119   460.0000
   H1                 Test  3576        NaN
   H2          Development 30485  3926.0000
   H2 Final refit eligible 29468  3882.0000
   H2 Final refit excluded  1017    44.0000
   H2          Calibration 12338  1417.0000
   H2                 Test  7748        NaN


The final refit membership excludes development bookings whose labels were not
yet available at the test start. Test bookings remain outside CV and refit
regardless of their eventual outcomes. Boundary and eligibility behavior is
covered in `tests/test_model_split.py`.

## 3. Model comparison on the same validation bookings

The initial Core logistic regression (21 inputs) shows ranking signal but poor
probabilities. Calendar ablation removes year, month and week; RF and XGBoost
then use the same 18 inputs. Tuning selects within each model family by mean
window Log Loss. The later RF feature experiment removes EURUSD to form F1.

In [5]:
baseline = report_table('baseline_v1')
calendar = report_table('lr_calendar_ablation_v1')
rf_initial = report_table('rf_initial_v1')
xgb_initial = report_table('xgb_initial_v1')
tuning = report_table('tuning_round1_v1')
tuning_summary = report_table('tuning_round1_v1', 'cv_summary.csv')
features = report_table('feature_ablation_round1_v1')

selected = [
    ('Dummy', baseline.loc[baseline.Model.eq('dummy')]),
    ('LR21 initial', baseline.loc[baseline.Model.eq('logistic_regression')]),
    ('LR18 initial', calendar.loc[calendar.FeatureSet.eq('core_no_year_month_week')]),
    ('RF18 initial', rf_initial),
    ('XGB18 initial', xgb_initial),
]
for model in ('LR18', 'RF18', 'XGB18'):
    winners = tuning_summary.loc[tuning_summary.ModelLabel.eq(model) &
                                 tuning_summary.BestWithinModel.astype(str).eq('True'),
                                 ['Hotel', 'ConfigID']]
    fold_rows = tuning.merge(winners, on=['Hotel', 'ConfigID'], validate='many_to_one')
    selected.append((model + ' tuned', fold_rows))
selected.append(('RF17 F1', features.loc[features.FeatureSet.eq('F1')]))

comparison = pd.concat([
    frame.groupby('Hotel', as_index=False)[METRICS].mean().assign(Variant=name)
    for name, frame in selected
], ignore_index=True)
comparison = comparison[['Hotel', 'Variant'] + METRICS].sort_values(['Hotel', 'LogLoss'])
show(comparison)

Hotel       Variant  LogLoss  BrierScore  ROC_AUC  AveragePrecision
   H1       RF17 F1   0.2649      0.0711   0.6576            0.1488
   H1  RF18 initial   0.2663      0.0716   0.6574            0.1504
   H1    RF18 tuned   0.2663      0.0716   0.6574            0.1504
   H1    LR18 tuned   0.2691      0.0722   0.6445            0.1306
   H1   XGB18 tuned   0.2693      0.0728   0.6498            0.1346
   H1  LR18 initial   0.2707      0.0728   0.6331            0.1245
   H1         Dummy   0.2772      0.0729   0.5000            0.0791
   H1 XGB18 initial   0.2815      0.0781   0.6466            0.1292
   H1  LR21 initial   0.5197      0.1742   0.6285            0.1219
   H2       RF17 F1   0.3575      0.1025   0.6127            0.1781
   H2    RF18 tuned   0.3595      0.1031   0.6072            0.1768
   H2         Dummy   0.3640      0.1039   0.5000            0.1166
   H2    LR18 tuned   0.3651      0.1050   0.5855            0.1457
   H2  RF18 initial   0.3670      0.1060   0.584

These are four-fold means, not pooled booking-level scores. Initial LR21 is
far worse than Dummy on Log Loss despite better ranking. The tuned RF improves
over Dummy for both hotels; its gain is small in H2. RF17 F1 has the lowest
mean Log Loss in this historical search: about 0.2649 for H1 and 0.3575 for H2.

### Cohort alignment gate

Every variant above must predict the same validation booking, label, fold and
date. This check is performed once for the comparison, rather than repeated in
each experiment section.

In [6]:
def historical_oof(run):
    return report_table(run, 'oof_predictions.csv')

reference = historical_oof('baseline_v1').query("Model == 'dummy'")
reference = reference[['Hotel', 'Booking_ID', 'Fold', 'PredictionDate', 'IsCanceled']]
assert not reference.duplicated(['Hotel', 'Booking_ID']).any()

oof_variants = [
    ('LR21', historical_oof('baseline_v1').query("Model == 'logistic_regression'")),
    ('LR18', historical_oof('lr_calendar_ablation_v1').query(
        "FeatureSet == 'core_no_year_month_week'")),
    ('RF18', historical_oof('rf_initial_v1')),
    ('XGB18', historical_oof('xgb_initial_v1')),
    ('RF17 F1', historical_oof('feature_ablation_round1_v1').query("FeatureSet == 'F1'")),
]
tuned_oof = historical_oof('tuning_round1_v1')
for model in ('LR18', 'RF18', 'XGB18'):
    winners = tuning_summary.loc[tuning_summary.ModelLabel.eq(model) &
                                 tuning_summary.BestWithinModel.astype(str).eq('True'),
                                 ['Hotel', 'ConfigID']]
    oof_variants.append((model + ' tuned', tuned_oof.merge(
        winners, on=['Hotel', 'ConfigID'], validate='many_to_one')))

for name, candidate in oof_variants:
    cohort = candidate[reference.columns]
    joined = cohort.merge(reference, on=['Hotel', 'Booking_ID'], how='outer',
                          suffixes=('_candidate', '_reference'), indicator=True,
                          validate='one_to_one')
    assert len(cohort) == len(reference) and joined._merge.eq('both').all(), name
    for column in ('Fold', 'PredictionDate', 'IsCanceled'):
        assert joined[f'{column}_candidate'].equals(joined[f'{column}_reference']), name
print(f'{len(oof_variants)} variants aligned with {len(reference):,} validation bookings')

8 variants aligned with 18,457 validation bookings


## 4. Diagnostics across windows

Fold averages can hide a weak window. The next tables expose the train versus
validation gap, cancellation-rate changes, and mean probability bias for the
current RF17 candidate. Positive `GainVsDummy` means lower Log Loss than Dummy.

### Why the Core logistic regression needed revision

Fold 1 is the clearest failure. Its calendar terms produce a strong upward
score shift while the most recent arrival cohort eligible at fit time is
dominated by early cancellations. The next table measures the probability
error; the full contribution analysis remains in the prior Git version.
This is a hypothesis about the model behavior, not proof that one mechanism
alone caused it.

In [7]:
baseline_oof = historical_oof('baseline_v1')
core_first = baseline_oof.loc[baseline_oof.Fold.eq(1)].groupby(
    ['Hotel', 'Model'], as_index=False).agg(
        Bookings=('Booking_ID', 'size'), ActualRate=('IsCanceled', 'mean'),
        MeanProbability=('RawProbability', 'mean'))
show(core_first)

Hotel               Model  Bookings  ActualRate  MeanProbability
   H1               dummy      1779      0.0646           0.0834
   H1 logistic_regression      1779      0.0646           0.6762
   H2               dummy      3805      0.0830           0.1535
   H2 logistic_regression      3805      0.0830           0.6228


At the 1 July fit, the July 2016 arrival cohort contributes only 15 eligible
H1 bookings and 22 eligible H2 bookings, all canceled or no-show; 295 and 784
bookings respectively are still unavailable. The chronological filter is
correct, but the newest observed cohort is heavily selected. Removing the
calendar inputs sharply improves fold-1 LR Log Loss.

In [8]:
f1 = features.loc[features.FeatureSet.eq('F1')].sort_values(['Hotel', 'Fold'])
show(f1[['Hotel', 'Fold', 'TrainRows', 'ValidationRows', 'TrainCancelRate',
         'ValidationCancelRate', 'MeanProbability', 'TrainLogLoss', 'LogLoss',
         'GeneralizationGap', 'GainVsDummy']])

Hotel  Fold  TrainRows  ValidationRows  TrainCancelRate  ValidationCancelRate  MeanProbability  TrainLogLoss  LogLoss  GeneralizationGap  GainVsDummy
   H1     1       8618            1779           0.0834                0.0646           0.0762        0.2192   0.2342             0.0150       0.0078
   H1     2      10353            1936           0.0797                0.0579           0.0761        0.2154   0.2034            -0.0119       0.0212
   H1     3      12291            1152           0.0769                0.0990           0.0973        0.2094   0.3160             0.1066       0.0100
   H1     4      13631            1252           0.0774                0.0950           0.0904        0.2115   0.3059             0.0944       0.0102
   H2     1      14230            3805           0.1535                0.0830           0.1552        0.3804   0.3038            -0.0767       0.0047
   H2     2      17962            3958           0.1401                0.1349           0.1386      

RF17 beats Dummy in all four windows for both hotels, but the margins vary.
H1's F1 improvement over RF18 is concentrated in later windows. The train to
validation gap remains material, so the mean gain alone does not establish
stable generalization.

### Feature and base-rate movement

These are descriptive shifts, not causal effects. `EURUSD` drops sharply in
later windows; lead-time mix and customer mix also move. The same booking can
have a different cancellation rate in a different window.

In [9]:
drift_rows = []
for hotel in HOTELS:
    dev = pd.read_parquet(ARTIFACTS / 'data' / hotel / 'dev.parquet')
    membership = pd.read_csv(ARTIFACTS / 'split' / hotel / 'cv_membership.csv',
                             dtype={'Booking_ID': str})
    dev['Booking_ID'] = dev['Booking_ID'].astype(str)
    for fold in sorted(membership.Fold.unique()):
        for role in ('Train', 'Validation'):
            keys = membership.loc[membership.Fold.eq(fold) & membership.Role.eq(role),
                                  ['Hotel', 'Booking_ID']]
            cohort = keys.merge(dev, on=['Hotel', 'Booking_ID'], validate='one_to_one')
            assert len(cohort) == len(keys)
            drift_rows.append({
                'Hotel': hotel, 'Fold': fold, 'Role': role, 'Rows': len(cohort),
                'CancelRate': cohort.IsCanceled.mean(),
                'MeanLeadTime': cohort.LeadTime.mean(),
                'MeanEURUSD': cohort.EURUSD.mean(),
                'TransientShare': cohort.CustomerType.eq('Transient').mean(),
            })
drift = pd.DataFrame(drift_rows)
show(drift)

Hotel  Fold       Role  Rows  CancelRate  MeanLeadTime  MeanEURUSD  TransientShare
   H1     1      Train  8618      0.0834      101.7520      1.1141          0.6317
   H1     1 Validation  1779      0.0646      138.9207      1.1145          0.8010
   H1     2      Train 10353      0.0797      106.9345      1.1139          0.6633
   H1     2 Validation  1936      0.0579      162.0723      1.1127          0.5785
   H1     3      Train 12291      0.0769      120.5455      1.1149          0.6432
   H1     3 Validation  1152      0.0990       71.7977      1.0675          0.6866
   H1     4      Train 13631      0.0774      115.1803      1.1118          0.6529
   H1     4 Validation  1252      0.0950       73.1222      1.0639          0.5447
   H2     1      Train 14230      0.1535       85.7330      1.1167          0.4665
   H2     1 Validation  3805      0.0830      130.7863      1.1147          0.7430
   H2     2      Train 17962      0.1401       94.9393      1.1159          0.5282
   H

### Where individual probability errors concentrate

The group table is diagnostic. Small groups can have unstable rates, so the
view below requires at least 30 bookings and ranks by contribution to excess
Log Loss versus Dummy within each hotel.

In [10]:
groups = pd.concat([pd.read_csv(ARTIFACTS / 'error_analysis' / hotel / 'group_errors.csv')
                    for hotel in HOTELS], ignore_index=True)
large_groups = groups.loc[groups.Rows.ge(30)]
hard_groups = large_groups.sort_values('CVExcessContribution', ascending=False).groupby(
    'Hotel', group_keys=False).head(6)
show(hard_groups[['Hotel', 'Fold', 'Dimension', 'Group', 'Rows', 'CancelRate',
                  'MeanProbability', 'ExcessLogLoss', 'CVExcessContribution']])

Hotel  Fold     Dimension     Group  Rows  CancelRate  MeanProbability  ExcessLogLoss  CVExcessContribution
   H2     3      LeadTime        G4   495      0.0788           0.1547         0.0364                0.0081
   H2     3 MarketSegment    Groups   246      0.2033           0.2219         0.0300                0.0033
   H1     4      LeadTime        G3   293      0.0512           0.0693         0.0125                0.0029
   H2     4      LeadTime        G1   949      0.1149           0.1715         0.0046                0.0018
   H1     3      LeadTime        G4   112      0.1250           0.0694         0.0150                0.0015
   H2     2 MarketSegment    Direct   252      0.0992           0.1487         0.0184                0.0012
   H2     1 MarketSegment    Direct   320      0.0750           0.1650         0.0116                0.0010
   H1     1 MarketSegment Online TA   858      0.0688           0.0952         0.0019                0.0009
   H2     4 MarketSegment Co

## 5. Targeted RF feature experiment

F0 is tuned RF with 18 inputs. F1 removes EURUSD, F2 removes LeadTime, and F3
removes both. Hyperparameters, eligible training rows and validation bookings
stay fixed within each hotel. This isolates the feature-set comparison within
the tested RF configuration.

In [11]:
feature_summary = report_table('feature_ablation_round1_v1', 'cv_summary.csv')
show(feature_summary[['Hotel', 'FeatureSet', 'Removed', 'InputColumns', 'LogLoss',
                      'BrierScore', 'GainVsF0', 'FoldsBeatingF0',
                      'MeanAbsoluteWindowBias']].sort_values(['Hotel', 'LogLoss']))

Hotel FeatureSet               Removed  InputColumns  LogLoss  BrierScore  GainVsF0  FoldsBeatingF0  MeanAbsoluteWindowBias
   H1         F1            ["EURUSD"]            17   0.2649      0.0711    0.0015               2                  0.0090
   H1         F0                    []            18   0.2663      0.0716    0.0000               0                  0.0201
   H1         F2          ["LeadTime"]            17   0.2689      0.0721   -0.0026               1                  0.0207
   H1         F3 ["EURUSD","LeadTime"]            16   0.2693      0.0718   -0.0029               1                  0.0119
   H2         F1            ["EURUSD"]            17   0.3575      0.1025    0.0021               4                  0.0320
   H2         F3 ["EURUSD","LeadTime"]            16   0.3576      0.1027    0.0019               3                  0.0308
   H2         F0                    []            18   0.3595      0.1031   -0.0000               0                  0.0374
   H2   

F1 has the lowest four-fold Log Loss for both hotels. The improvement over F0
is small, especially in H1, where it comes mainly from fold 4. Removing
LeadTime alone worsens mean Log Loss for both. This supports carrying F1
forward as a candidate; it does not prove EURUSD is harmful in every setting.

## 6. Probability calibration

Sigmoid and isotonic mappings use earlier-fold OOF predictions and are scored
on future folds 2–4. The raw reference in this section uses **those same three
folds**; do not compare its mean directly with a four-fold score above.

In [12]:
calibration_summary = pd.read_csv(REPORTS / 'calibration_round1_v1' / 'cv_summary.csv')
calibration_f1 = calibration_summary.loc[calibration_summary.FeatureSet.eq('F1')]
show(calibration_f1[['Hotel', 'Method', 'EvaluationFolds', 'LogLoss', 'BrierScore',
                     'GainVsRaw', 'FoldsBeatingRaw', 'MeanAbsoluteWindowBias',
                     'ExactZeroRows']].sort_values(['Hotel', 'LogLoss']))

Hotel   Method EvaluationFolds  LogLoss  BrierScore  GainVsRaw  FoldsBeatingRaw  MeanAbsoluteWindowBias  ExactZeroRows
   H1      raw           2,3,4   0.2751      0.0748     0.0000                0                  0.0082              0
   H1  sigmoid           2,3,4   0.2791      0.0757    -0.0040                0                  0.0141              0
   H1 isotonic           2,3,4   0.3038      0.0757    -0.0287                0                  0.0126            131
   H2      raw           2,3,4   0.3754      0.1099    -0.0000                0                  0.0187              0
   H2  sigmoid           2,3,4   0.3782      0.1105    -0.0028                1                  0.0189              0
   H2 isotonic           2,3,4   0.4059      0.1106    -0.0305                1                  0.0225            388


In [13]:
calibration_folds = pd.read_csv(REPORTS / 'calibration_round1_v1' / 'fold_metrics.csv')
show(calibration_folds.loc[calibration_folds.FeatureSet.eq('F1'),
                           ['Hotel', 'Fold', 'Method', 'ValidationCancelRate',
                            'MeanProbability', 'ProbabilityBias', 'LogLoss',
                            'GainVsRaw', 'ExactZeroRows']].sort_values(
                                ['Hotel', 'Fold', 'Method']))

Hotel  Fold   Method  ValidationCancelRate  MeanProbability  ProbabilityBias  LogLoss  GainVsRaw  ExactZeroRows
   H1     2 isotonic                0.0579           0.0829           0.0250   0.2068    -0.0033             50
   H1     2      raw                0.0579           0.0761           0.0183   0.2034     0.0000              0
   H1     2  sigmoid                0.0579           0.0797           0.0219   0.2061    -0.0027              0
   H1     3 isotonic                0.0990           0.0979          -0.0011   0.3165    -0.0005             30
   H1     3      raw                0.0990           0.0973          -0.0016   0.3160     0.0000              0
   H1     3  sigmoid                0.0990           0.0953          -0.0036   0.3234    -0.0074              0
   H1     4 isotonic                0.0950           0.0833          -0.0118   0.3881    -0.0823             51
   H1     4      raw                0.0950           0.0904          -0.0047   0.3059     0.0000        

Raw F1 has the lowest mean Log Loss and Brier score under this calibration
protocol for both hotels. In H2 fold 2, calibration moves the mean probability
away from the observed rate. Isotonic also assigns zero probability to some
bookings that cancel, causing large individual Log Loss penalties. The
evidence favors keeping raw probabilities for the current candidate.

## 7. Decision and remaining gate

**Current candidate:** tuned RF with F1 (17 inputs), expanding eligible
training history, and raw probabilities for both hotels. It has the lowest
four-fold mean Log Loss in the historical comparison, with modest gains.

Before final refit, confirm that all 17 inputs can be produced at the T-14
prediction cutoff, record the acceptance rule, and rerun the parity check after
any pipeline change. After freezing the selection,
run the DVC `final` stage once for the held-out test. No test metric is used
in this notebook.